# NL2SQL Graph 테스트 노트북

이 노트북은 NL2SQL Graph의 각 컴포넌트를 독립적으로 테스트할 수 있도록 구성되었습니다.

## 목차
1. 환경 설정 및 Mock 데이터 준비
2. LLM 연결 테스트
3. SQL 생성 노드 테스트
4. SQL 검증 노드 테스트
5. SQL 실행 노드 테스트 (Mock DB)
6. 답변 생성 노드 테스트
7. 전체 Graph 통합 테스트

## 1. 환경 설정 및 Mock 데이터 준비

## 0. 패키지 설치 (최초 1회만 실행)

In [1]:
# 필수 라이브러리 임포트
import os
import sys
from typing import Any, Dict, List, TypedDict
from unittest.mock import Mock, patch
import json

# 프로젝트 루트 경로 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

# LangChain 및 LangGraph 임포트
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

print("✅ 라이브러리 임포트 완료")
print(f"프로젝트 경로: {project_root}")

✅ 라이브러리 임포트 완료
프로젝트 경로: d:\900.develop\02.dev\30.python\12.hr-chatbot-claude


In [2]:
# 환경 변수 설정 (실제 OpenAI API 키 필요)
from dotenv import load_dotenv

# .env 파일 로드
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

# API 키 확인
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"✅ OpenAI API Key 로드 완료: {openai_api_key[:10]}...")
else:
    print("⚠️ OpenAI API Key가 설정되지 않았습니다. Mock LLM을 사용합니다.")

✅ OpenAI API Key 로드 완료: sk-proj-s7...


In [3]:
# Mock 데이터베이스 스키마 정의
MOCK_SCHEMA = """
# HR 데이터베이스 스키마

## 테이블: employee
컬럼:
  - id: integer (NOT NULL)
  - emp_id: character varying (NOT NULL)
  - name: character varying (NOT NULL)
  - email: character varying (NOT NULL)
  - hire_date: date (NOT NULL)
  - position: character varying (NOT NULL)
  - job_family: character varying (NOT NULL)
  - work_location: character varying (NOT NULL)
  - status: character varying (NOT NULL)
  - department_id: integer (NULL)
기본 키: id
외래 키:
  - department_id -> department.id
샘플 데이터 (예시):
  1. {'id': 1, 'emp_id': 'EMP001', 'name': '김철수', 'email': 'kim@example.com', 'hire_date': '2024-01-15', 'position': '사원', 'job_family': '개발', 'work_location': '서울', 'status': 'active', 'department_id': 1}
  2. {'id': 2, 'emp_id': 'EMP002', 'name': '이영희', 'email': 'lee@example.com', 'hire_date': '2024-02-20', 'position': '대리', 'job_family': '기획', 'work_location': '부산', 'status': 'active', 'department_id': 2}

## 테이블: department
컬럼:
  - id: integer (NOT NULL)
  - dept_code: character varying (NOT NULL)
  - name: character varying (NOT NULL)
  - description: text (NULL)
기본 키: id
샘플 데이터 (예시):
  1. {'id': 1, 'dept_code': 'DEV', 'name': '개발팀', 'description': '소프트웨어 개발'}
  2. {'id': 2, 'dept_code': 'PLAN', 'name': '기획팀', 'description': '제품 기획'}
"""

print("✅ Mock 스키마 정의 완료")
print(MOCK_SCHEMA)

✅ Mock 스키마 정의 완료

# HR 데이터베이스 스키마

## 테이블: employee
컬럼:
  - id: integer (NOT NULL)
  - emp_id: character varying (NOT NULL)
  - name: character varying (NOT NULL)
  - email: character varying (NOT NULL)
  - hire_date: date (NOT NULL)
  - position: character varying (NOT NULL)
  - job_family: character varying (NOT NULL)
  - work_location: character varying (NOT NULL)
  - status: character varying (NOT NULL)
  - department_id: integer (NULL)
기본 키: id
외래 키:
  - department_id -> department.id
샘플 데이터 (예시):
  1. {'id': 1, 'emp_id': 'EMP001', 'name': '김철수', 'email': 'kim@example.com', 'hire_date': '2024-01-15', 'position': '사원', 'job_family': '개발', 'work_location': '서울', 'status': 'active', 'department_id': 1}
  2. {'id': 2, 'emp_id': 'EMP002', 'name': '이영희', 'email': 'lee@example.com', 'hire_date': '2024-02-20', 'position': '대리', 'job_family': '기획', 'work_location': '부산', 'status': 'active', 'department_id': 2}

## 테이블: department
컬럼:
  - id: integer (NOT NULL)
  - dept_code: character vary

In [4]:
# Mock SQL 실행 결과 데이터
MOCK_SQL_RESULTS = {
    "count_query": {
        "columns": ["count"],
        "rows": [[27]],
        "row_count": 1,
        "execution_time_ms": 15
    },
    "employee_list": {
        "columns": ["emp_id", "name", "position", "hire_date"],
        "rows": [
            ["EMP001", "김철수", "사원", "2024-01-15"],
            ["EMP002", "이영희", "대리", "2024-02-20"],
            ["EMP003", "박민수", "과장", "2024-03-10"]
        ],
        "row_count": 3,
        "execution_time_ms": 23
    },
    "department_stats": {
        "columns": ["department", "count"],
        "rows": [
            ["개발팀", 15],
            ["기획팀", 8],
            ["디자인팀", 4]
        ],
        "row_count": 3,
        "execution_time_ms": 18
    }
}

print("✅ Mock SQL 결과 데이터 정의 완료")
print(json.dumps(MOCK_SQL_RESULTS, indent=2, ensure_ascii=False))

✅ Mock SQL 결과 데이터 정의 완료
{
  "count_query": {
    "columns": [
      "count"
    ],
    "rows": [
      [
        27
      ]
    ],
    "row_count": 1,
    "execution_time_ms": 15
  },
  "employee_list": {
    "columns": [
      "emp_id",
      "name",
      "position",
      "hire_date"
    ],
    "rows": [
      [
        "EMP001",
        "김철수",
        "사원",
        "2024-01-15"
      ],
      [
        "EMP002",
        "이영희",
        "대리",
        "2024-02-20"
      ],
      [
        "EMP003",
        "박민수",
        "과장",
        "2024-03-10"
      ]
    ],
    "row_count": 3,
    "execution_time_ms": 23
  },
  "department_stats": {
    "columns": [
      "department",
      "count"
    ],
    "rows": [
      [
        "개발팀",
        15
      ],
      [
        "기획팀",
        8
      ],
      [
        "디자인팀",
        4
      ]
    ],
    "row_count": 3,
    "execution_time_ms": 18
  }
}


In [5]:
# NL2SQL State 정의 (원본 코드와 동일)
class NL2SQLState(TypedDict):
    """NL2SQL Graph 상태"""
    question: str
    schema_description: str
    generated_sql: str
    validated: bool
    validation_error: str
    sql_result: Dict[str, Any]  # Mock용으로 간소화
    answer: str
    metadata: Dict[str, Any]
    request_id: str

print("✅ NL2SQLState TypedDict 정의 완료")

✅ NL2SQLState TypedDict 정의 완료


In [6]:
# 로깅 헬퍼 함수
def log_step(request_id: str, step: str, stage: str, message: str, **kwargs):
    """간단한 로그 출력 함수"""
    extra = " | ".join([f"{k}={v}" for k, v in kwargs.items()]) if kwargs else ""
    print(f"[{request_id}] [NL2SQL-{step}] [{stage}] {message}" + (f" | {extra}" if extra else ""))

def truncate_text(text: str, max_length: int = 200) -> str:
    """텍스트 자르기"""
    if not text:
        return ""
    text = text.replace("\n", " ").strip()
    if len(text) <= max_length:
        return text
    return text[:max_length] + "...[truncated]"

print("✅ 로깅 헬퍼 함수 정의 완료")

✅ 로깅 헬퍼 함수 정의 완료


## 2. LLM 연결 테스트

In [28]:
# 실제 LLM 연결 테스트 (API 키가 있는 경우)
if openai_api_key:
    try:
        llm = ChatOpenAI(
            model="gpt-4.1-nano",  # 저렴한 모델 사용
            temperature=0,
            api_key=openai_api_key
        )
        
        # 간단한 테스트
        test_messages = [
            SystemMessage(content="You are a helpful assistant."),
            HumanMessage(content="Say 'Hello, NL2SQL!'")
        ]
        
        print(f"요청 메시지: {test_messages}")
        response = llm.invoke(test_messages)
        print("✅ LLM 연결 성공!")
        print(f"응답: {response.content}")
        
        USE_REAL_LLM = True
    except Exception as e:
        print(f"❌ LLM 연결 실패: {e}")
        print("Mock LLM을 사용합니다.")
        USE_REAL_LLM = False
else:
    print("⚠️ API 키가 없어 Mock LLM을 사용합니다.")
    USE_REAL_LLM = False
    
 

요청 메시지: [SystemMessage(content='You are a helpful assistant.'), HumanMessage(content="Say 'Hello, NL2SQL!'")]
✅ LLM 연결 성공!
응답: Hello, NL2SQL!


In [20]:
response

AIMessage(content='Hello, NL2SQL!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 25, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_f0bc439dc3', 'finish_reason': 'stop', 'logprobs': None}, id='run-8f4abe95-2aca-4d87-b4c5-c5233f8c0f65-0', usage_metadata={'input_tokens': 25, 'output_tokens': 6, 'total_tokens': 31})

In [29]:
# Mock LLM 클래스 정의
class MockLLM:
    """테스트용 Mock LLM"""
    
    def __init__(self, model="mock-gpt-4", temperature=0):
        self.model_name = model
        self.temperature = temperature
    
    def invoke(self, messages):
        """메시지에 따라 미리 정의된 응답 반환"""
        # 마지막 메시지 내용 확인
        user_message = messages[-1].content if messages else ""
        
        # SQL 생성 요청인 경우
        if "SQL 쿼리를 생성" in user_message or "PostgreSQL SELECT" in user_message:
            if "2024년 입사자" in user_message or "2024년에 입사한" in user_message:
                sql = "SELECT emp_id, name, hire_date FROM employee WHERE EXTRACT(YEAR FROM hire_date) = 2024 LIMIT 1000;"
            elif "개발팀" in user_message:
                sql = "SELECT e.emp_id, e.name, e.position FROM employee e JOIN department d ON e.department_id = d.id WHERE d.name = '개발팀' LIMIT 1000;"
            elif "몇 명" in user_message or "총 수" in user_message:
                sql = "SELECT COUNT(*) as count FROM employee WHERE status = 'active';"
            else:
                sql = "SELECT * FROM employee LIMIT 10;"
            
            return AIMessage(content=sql)
        
        # 답변 생성 요청인 경우
        elif "답변을 자연어" in user_message or "질문에 대한 답변" in user_message:
            return AIMessage(content="조회 결과를 바탕으로 2024년에 입사한 직원은 총 3명입니다. 김철수(EMP001), 이영희(EMP002), 박민수(EMP003)입니다.")
        
        # 기본 응답
        return AIMessage(content="Mock LLM Response")

# Mock LLM 인스턴스 생성
if not USE_REAL_LLM:
    llm = MockLLM()
    print("✅ Mock LLM 생성 완료")
    
    # Mock LLM 테스트
    test_response = llm.invoke([
        HumanMessage(content="질문: 2024년에 입사한 직원은?\n\n위 질문에 대한 PostgreSQL SELECT 쿼리를 생성해주세요.")
    ])
    print(f"Mock LLM 응답: {test_response.content}")

## 3. SQL 생성 노드 테스트

In [30]:
# SQL 생성 노드 함수
def generate_sql_node(state: NL2SQLState, llm_instance, schema: str) -> NL2SQLState:
    """SQL 생성 노드"""
    question = state["question"]
    request_id = state.get("request_id", "unknown")
    
    log_step(request_id, "1", "GENERATE", "SQL 생성 시작", question=question[:40])
    
    # 시스템 프롬프트
    system_prompt = f"""당신은 PostgreSQL 전문가입니다.
사용자의 자연어 질문을 PostgreSQL SQL 쿼리로 변환해주세요.

# 데이터베이스 스키마
{schema}

# 중요한 규칙
1. **반드시 SELECT 문만 생성하세요** (INSERT, UPDATE, DELETE, DROP 등은 절대 사용 금지)
2. **테이블명과 컬럼명은 정확하게 사용하세요**
3. **SQL만 출력하고, 설명이나 마크다운 코드 블록은 포함하지 마세요**
"""
    
    user_prompt = f"""질문: {question}

위 질문에 대한 PostgreSQL SELECT 쿼리를 생성해주세요.
SQL만 출력하세요 (설명 없이)."""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    
    log_step(request_id, "1a", "LLM-INPUT", "LLM 호출 시작")
    
    try:
        response = llm_instance.invoke(messages)
        sql = response.content.strip()
        
        # 마크다운 코드 블록 제거
        if sql.startswith("```"):
            lines = sql.split("\n")
            sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
            sql = sql.replace("```sql", "").replace("```", "").strip()
        
        state["generated_sql"] = sql
        state["schema_description"] = schema
        state["metadata"]["llm_model"] = getattr(llm_instance, 'model_name', 'unknown')
        
        log_step(request_id, "1b", "LLM-OUTPUT", "SQL 생성 완료", sql_length=len(sql))
        log_step(request_id, "1b", "LLM-OUTPUT", f"GENERATED_SQL: {truncate_text(sql)}")
        
    except Exception as e:
        print(f"❌ SQL 생성 실패: {e}")
        state["generated_sql"] = ""
        state["validation_error"] = f"SQL 생성 오류: {str(e)}"
        state["validated"] = False
    
    return state

print("✅ SQL 생성 노드 함수 정의 완료")

✅ SQL 생성 노드 함수 정의 완료


In [31]:
# SQL 생성 노드 테스트
test_state_1 = NL2SQLState(
    question="2024년에 입사한 직원은 몇 명인가요?",
    schema_description="",
    generated_sql="",
    validated=False,
    validation_error="",
    sql_result={},
    answer="",
    metadata={},
    request_id="test-001"
)

print("\n" + "="*80)
print("SQL 생성 노드 테스트 시작")
print("="*80)

result_state_1 = generate_sql_node(test_state_1, llm, MOCK_SCHEMA)

print("\n" + "="*80)
print("생성된 SQL:")
print("="*80)
print(result_state_1["generated_sql"])
print("\n")


SQL 생성 노드 테스트 시작
[test-001] [NL2SQL-1] [GENERATE] SQL 생성 시작 | question=2024년에 입사한 직원은 몇 명인가요?
[test-001] [NL2SQL-1a] [LLM-INPUT] LLM 호출 시작
[test-001] [NL2SQL-1b] [LLM-OUTPUT] SQL 생성 완료 | sql_length=72
[test-001] [NL2SQL-1b] [LLM-OUTPUT] GENERATED_SQL: SELECT COUNT(*) FROM employee WHERE EXTRACT(YEAR FROM hire_date) = 2024;

생성된 SQL:
SELECT COUNT(*) FROM employee WHERE EXTRACT(YEAR FROM hire_date) = 2024;




## 4. SQL 검증 노드 테스트

In [32]:
# SQL 검증 함수 (간소화 버전)
import sqlparse
from sqlparse.sql import Statement
from sqlparse.tokens import Keyword, DML

def validate_sql_simple(sql: str) -> tuple[bool, str]:
    """간단한 SQL 검증"""
    if not sql or not sql.strip():
        return False, "SQL이 비어있습니다"
    
    # 위험한 키워드 확인
    dangerous_keywords = [
        'DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 
        'CREATE', 'TRUNCATE', 'GRANT', 'REVOKE'
    ]
    
    sql_upper = sql.upper()
    for keyword in dangerous_keywords:
        if keyword in sql_upper:
            return False, f"허용되지 않는 키워드: {keyword}"
    
    # SELECT 문인지 확인
    if not sql_upper.strip().startswith('SELECT'):
        return False, "SELECT 문만 허용됩니다"
    
    # 허용된 테이블 확인 (간소화)
    allowed_tables = ['employee', 'department', 'hr_docs']
    
    return True, ""

print("✅ SQL 검증 함수 정의 완료")

✅ SQL 검증 함수 정의 완료


In [33]:
# SQL 검증 노드 함수
def validate_sql_node(state: NL2SQLState) -> NL2SQLState:
    """SQL 검증 노드"""
    sql = state["generated_sql"]
    request_id = state.get("request_id", "unknown")
    
    if not sql:
        log_step(request_id, "2", "VALIDATE", "검증 실패 - SQL 없음")
        state["validated"] = False
        state["validation_error"] = "생성된 SQL이 없습니다"
        return state
    
    log_step(request_id, "2", "VALIDATE", "SQL 검증 시작")
    
    try:
        is_valid, error_msg = validate_sql_simple(sql)
        
        if is_valid:
            state["validated"] = True
            state["validation_error"] = ""
            log_step(request_id, "2", "VALIDATE", "SQL 검증 성공")
        else:
            state["validated"] = False
            state["validation_error"] = error_msg
            log_step(request_id, "2", "VALIDATE", f"SQL 검증 실패", error=error_msg[:50])
    
    except Exception as e:
        state["validated"] = False
        state["validation_error"] = str(e)
        print(f"❌ SQL 검증 중 오류: {e}")
    
    return state

print("✅ SQL 검증 노드 함수 정의 완료")

✅ SQL 검증 노드 함수 정의 완료


In [34]:
# SQL 검증 노드 테스트
print("\n" + "="*80)
print("SQL 검증 노드 테스트 시작")
print("="*80)

result_state_2 = validate_sql_node(result_state_1)

print("\n" + "="*80)
print(f"검증 결과: {'✅ 성공' if result_state_2['validated'] else '❌ 실패'}")
if result_state_2['validation_error']:
    print(f"오류 메시지: {result_state_2['validation_error']}")
print("="*80 + "\n")


SQL 검증 노드 테스트 시작
[test-001] [NL2SQL-2] [VALIDATE] SQL 검증 시작
[test-001] [NL2SQL-2] [VALIDATE] SQL 검증 성공

검증 결과: ✅ 성공



In [35]:
# 위험한 SQL 검증 테스트
dangerous_sql_tests = [
    "DROP TABLE employee;",
    "DELETE FROM employee WHERE id = 1;",
    "UPDATE employee SET name = 'hacked';",
    "INSERT INTO employee VALUES (999, 'test');"
]

print("\n" + "="*80)
print("위험한 SQL 검증 테스트")
print("="*80)

for dangerous_sql in dangerous_sql_tests:
    is_valid, error = validate_sql_simple(dangerous_sql)
    status = "❌ 차단됨" if not is_valid else "⚠️ 통과 (문제!)"
    print(f"{status}: {dangerous_sql[:50]}")
    if error:
        print(f"  → {error}")
    print()


위험한 SQL 검증 테스트
❌ 차단됨: DROP TABLE employee;
  → 허용되지 않는 키워드: DROP

❌ 차단됨: DELETE FROM employee WHERE id = 1;
  → 허용되지 않는 키워드: DELETE

❌ 차단됨: UPDATE employee SET name = 'hacked';
  → 허용되지 않는 키워드: UPDATE

❌ 차단됨: INSERT INTO employee VALUES (999, 'test');
  → 허용되지 않는 키워드: INSERT



## 5. SQL 실행 노드 테스트 (Mock DB)

In [36]:
# Mock SQL 실행기
def execute_sql_mock(sql: str) -> Dict[str, Any]:
    """Mock SQL 실행 - 미리 정의된 결과 반환"""
    sql_lower = sql.lower()
    
    # COUNT 쿼리인 경우
    if 'count(*)' in sql_lower or 'count(' in sql_lower:
        return MOCK_SQL_RESULTS["count_query"]
    
    # JOIN 쿼리 또는 부서 관련
    elif 'join' in sql_lower or 'department' in sql_lower:
        return MOCK_SQL_RESULTS["department_stats"]
    
    # 기본 직원 목록
    else:
        return MOCK_SQL_RESULTS["employee_list"]

print("✅ Mock SQL 실행기 정의 완료")

✅ Mock SQL 실행기 정의 완료


In [37]:
# SQL 실행 노드 함수
def execute_sql_node(state: NL2SQLState) -> NL2SQLState:
    """SQL 실행 노드"""
    sql = state["generated_sql"]
    request_id = state.get("request_id", "unknown")
    
    log_step(request_id, "3", "EXECUTE", "SQL 실행 시작")
    
    try:
        result = execute_sql_mock(sql)
        state["sql_result"] = result
        state["metadata"]["execution_time_ms"] = result["execution_time_ms"]
        state["metadata"]["row_count"] = result["row_count"]
        
        log_step(request_id, "3", "EXECUTE", "SQL 실행 완료", 
                row_count=result["row_count"], 
                execution_time_ms=result["execution_time_ms"])
        
    except Exception as e:
        print(f"❌ SQL 실행 실패: {e}")
        state["validation_error"] = str(e)
        state["validated"] = False
    
    return state

print("✅ SQL 실행 노드 함수 정의 완료")

✅ SQL 실행 노드 함수 정의 완료


In [38]:
# SQL 실행 노드 테스트
print("\n" + "="*80)
print("SQL 실행 노드 테스트 시작")
print("="*80)

result_state_3 = execute_sql_node(result_state_2)

print("\n" + "="*80)
print("실행 결과:")
print("="*80)
print(json.dumps(result_state_3["sql_result"], indent=2, ensure_ascii=False))
print("\n")


SQL 실행 노드 테스트 시작
[test-001] [NL2SQL-3] [EXECUTE] SQL 실행 시작
[test-001] [NL2SQL-3] [EXECUTE] SQL 실행 완료 | row_count=1 | execution_time_ms=15

실행 결과:
{
  "columns": [
    "count"
  ],
  "rows": [
    [
      27
    ]
  ],
  "row_count": 1,
  "execution_time_ms": 15
}




## 6. 답변 생성 노드 테스트

In [39]:
# 답변 생성 노드 함수
def generate_answer_node(state: NL2SQLState, llm_instance) -> NL2SQLState:
    """답변 생성 노드"""
    question = state["question"]
    sql = state["generated_sql"]
    result = state.get("sql_result")
    request_id = state.get("request_id", "unknown")
    
    if not result or result.get("row_count", 0) == 0:
        log_step(request_id, "4", "ANSWER", "결과 없음 - 기본 응답 반환")
        state["answer"] = "조회된 결과가 없습니다."
        return state
    
    log_step(request_id, "4", "ANSWER", "답변 생성 시작", row_count=result["row_count"])
    
    # 시스템 프롬프트
    system_prompt = """당신은 데이터 분석 전문가입니다.
SQL 쿼리 결과를 사용자가 이해하기 쉽게 자연어로 요약해주세요.

답변 작성 시:
1. 핵심 통계나 수치를 강조하세요
2. 결과를 명확하고 간결하게 설명하세요
3. 필요시 불릿 포인트를 사용하세요
"""
    
    # 결과 데이터 요약
    rows_summary = result["rows"][:10] if len(result["rows"]) > 10 else result["rows"]
    
    user_prompt = f"""질문: {question}

실행된 SQL:
{sql}

조회 결과 ({result['row_count']}개 행):
컬럼: {', '.join(result['columns'])}
데이터 (샘플):
{rows_summary}

위 결과를 바탕으로 질문에 대한 답변을 자연어로 작성하세요."""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    
    log_step(request_id, "4a", "LLM-INPUT", "LLM 호출 시작 (답변 생성)")
    
    try:
        response = llm_instance.invoke(messages)
        answer = response.content
        
        state["answer"] = answer
        
        log_step(request_id, "4b", "LLM-OUTPUT", "답변 생성 완료", answer_length=len(answer))
        log_step(request_id, "4b", "LLM-OUTPUT", f"ANSWER: {truncate_text(answer)}")
        
    except Exception as e:
        print(f"❌ 답변 생성 실패: {e}")
        state["answer"] = f"조회 결과: {result['row_count']}개 행이 발견되었습니다."
    
    return state

print("✅ 답변 생성 노드 함수 정의 완료")

✅ 답변 생성 노드 함수 정의 완료


In [40]:
# 답변 생성 노드 테스트
print("\n" + "="*80)
print("답변 생성 노드 테스트 시작")
print("="*80)

result_state_4 = generate_answer_node(result_state_3, llm)

print("\n" + "="*80)
print("최종 답변:")
print("="*80)
print(result_state_4["answer"])
print("\n")


답변 생성 노드 테스트 시작
[test-001] [NL2SQL-4] [ANSWER] 답변 생성 시작 | row_count=1
[test-001] [NL2SQL-4a] [LLM-INPUT] LLM 호출 시작 (답변 생성)
[test-001] [NL2SQL-4b] [LLM-OUTPUT] 답변 생성 완료 | answer_length=24
[test-001] [NL2SQL-4b] [LLM-OUTPUT] ANSWER: 2024년에 입사한 직원은 총 27명입니다.

최종 답변:
2024년에 입사한 직원은 총 27명입니다.




## 7. 전체 Graph 통합 테스트

In [41]:
# 조건부 엣지 함수
def should_execute(state: NL2SQLState) -> str:
    """검증 성공 시 execute, 실패 시 error"""
    request_id = state.get("request_id", "unknown")
    decision = "execute" if state["validated"] else "error"
    log_step(request_id, "2x", "BRANCH", f"분기 결정 → {decision.upper()}")
    return decision

# 에러 핸들러
def handle_error_node(state: NL2SQLState) -> NL2SQLState:
    """에러 처리 노드"""
    error_msg = state.get("validation_error", "알 수 없는 오류")
    request_id = state.get("request_id", "unknown")
    
    state["answer"] = f"""SQL 생성 또는 실행 중 오류가 발생했습니다.

오류 내용: {error_msg}

다음 사항을 확인해주세요:
1. 질문이 데이터베이스 스키마에 맞는지 확인
2. 테이블명과 컬럼명이 정확한지 확인
3. 질문을 더 구체적으로 작성
"""
    
    log_step(request_id, "ERR", "ERROR", f"오류 처리 완료", error=error_msg[:50])
    return state

print("✅ 조건부 엣지 및 에러 핸들러 정의 완료")

✅ 조건부 엣지 및 에러 핸들러 정의 완료


In [45]:
# NL2SQL Graph 클래스 정의 (간소화 버전)
class NL2SQLGraphTest:
    """테스트용 NL2SQL Graph"""
    
    def __init__(self, llm_instance, schema: str):
        self.llm = llm_instance
        self.schema = schema
        self.graph = self._build_graph()
    
    def _build_graph(self):
        """그래프 구성"""
        workflow = StateGraph(NL2SQLState)
        
        # 노드 추가 (람다로 wrapping)
        workflow.add_node("generate_sql", lambda state: generate_sql_node(state, self.llm, self.schema))
        workflow.add_node("validate_sql", validate_sql_node)
        workflow.add_node("execute_sql", execute_sql_node)
        workflow.add_node("generate_answer", lambda state: generate_answer_node(state, self.llm))
        workflow.add_node("handle_error", handle_error_node)
        
        # 엣지 정의
        workflow.set_entry_point("generate_sql")
        workflow.add_edge("generate_sql", "validate_sql")
        
        # 조건부 엣지
        workflow.add_conditional_edges(
            "validate_sql",
            should_execute,
            {
                "execute": "execute_sql",
                "error": "handle_error"
            }
        )
        
        workflow.add_edge("execute_sql", "generate_answer")
        workflow.add_edge("generate_answer", END)
        workflow.add_edge("handle_error", END)
        
        return workflow.compile()
    
    def invoke(self, question: str, request_id: str = "test"):
        """동기 실행"""
        initial_state = NL2SQLState(
            question=question,
            schema_description="",
            generated_sql="",
            validated=False,
            validation_error="",
            sql_result={},
            answer="",
            metadata={},
            request_id=request_id
        )
        
        log_step(request_id, "0", "INIT", "NL2SQL 그래프 실행 시작", question=question[:40])
        
        result = self.graph.invoke(initial_state)
        
        log_step(request_id, "5", "COMPLETE", "NL2SQL 그래프 실행 완료", 
                has_sql=bool(result["generated_sql"]), 
                answer_length=len(result["answer"]))
        
        return result

print("✅ NL2SQLGraphTest 클래스 정의 완료")

✅ NL2SQLGraphTest 클래스 정의 완료


In [46]:
# Graph 인스턴스 생성
nl2sql_graph_test = NL2SQLGraphTest(llm, MOCK_SCHEMA)
print("✅ NL2SQL Graph 인스턴스 생성 완료")

✅ NL2SQL Graph 인스턴스 생성 완료


In [47]:
# 테스트 케이스 1: 정상적인 질문
print("\n" + "="*100)
print("테스트 케이스 1: 2024년 입사자 조회")
print("="*100 + "\n")

result1 = nl2sql_graph_test.invoke(
    question="2024년에 입사한 직원 목록을 보여주세요",
    request_id="tc-001"
)

print("\n" + "-"*100)
print("최종 결과:")
print("-"*100)
print(f"생성된 SQL:\n{result1['generated_sql']}\n")
print(f"검증 결과: {'✅ 통과' if result1['validated'] else '❌ 실패'}")
print(f"\n최종 답변:\n{result1['answer']}")
print("\n" + "="*100 + "\n")


테스트 케이스 1: 2024년 입사자 조회

[tc-001] [NL2SQL-0] [INIT] NL2SQL 그래프 실행 시작 | question=2024년에 입사한 직원 목록을 보여주세요
[tc-001] [NL2SQL-1] [GENERATE] SQL 생성 시작 | question=2024년에 입사한 직원 목록을 보여주세요
[tc-001] [NL2SQL-1a] [LLM-INPUT] LLM 호출 시작
[tc-001] [NL2SQL-1b] [LLM-OUTPUT] SQL 생성 완료 | sql_length=84
[tc-001] [NL2SQL-1b] [LLM-OUTPUT] GENERATED_SQL: SELECT * FROM employee WHERE hire_date >= '2024-01-01' AND hire_date < '2025-01-01';
[tc-001] [NL2SQL-2] [VALIDATE] SQL 검증 시작
[tc-001] [NL2SQL-2] [VALIDATE] SQL 검증 성공
[tc-001] [NL2SQL-2x] [BRANCH] 분기 결정 → EXECUTE
[tc-001] [NL2SQL-3] [EXECUTE] SQL 실행 시작
[tc-001] [NL2SQL-3] [EXECUTE] SQL 실행 완료 | row_count=3 | execution_time_ms=23
[tc-001] [NL2SQL-4] [ANSWER] 답변 생성 시작 | row_count=3
[tc-001] [NL2SQL-4a] [LLM-INPUT] LLM 호출 시작 (답변 생성)
[tc-001] [NL2SQL-4b] [LLM-OUTPUT] 답변 생성 완료 | answer_length=160
[tc-001] [NL2SQL-4b] [LLM-OUTPUT] ANSWER: 2024년에 입사한 직원은 총 3명입니다. 이들은 각각 다음과 같습니다:  - 김철수, 사원, 2024년 1월 15일 입사 - 이영희, 대리, 2024년 2월 20일 입사 - 박민수, 과장, 2024년 3월 10일  이들 직원은 모

In [48]:
# 테스트 케이스 2: 집계 쿼리
print("\n" + "="*100)
print("테스트 케이스 2: 재직 중인 직원 수 조회")
print("="*100 + "\n")

result2 = nl2sql_graph_test.invoke(
    question="현재 재직 중인 직원은 몇 명인가요?",
    request_id="tc-002"
)

print("\n" + "-"*100)
print("최종 결과:")
print("-"*100)
print(f"생성된 SQL:\n{result2['generated_sql']}\n")
print(f"검증 결과: {'✅ 통과' if result2['validated'] else '❌ 실패'}")
print(f"\n최종 답변:\n{result2['answer']}")
print("\n" + "="*100 + "\n")


테스트 케이스 2: 재직 중인 직원 수 조회

[tc-002] [NL2SQL-0] [INIT] NL2SQL 그래프 실행 시작 | question=현재 재직 중인 직원은 몇 명인가요?
[tc-002] [NL2SQL-1] [GENERATE] SQL 생성 시작 | question=현재 재직 중인 직원은 몇 명인가요?
[tc-002] [NL2SQL-1a] [LLM-INPUT] LLM 호출 시작
[tc-002] [NL2SQL-1b] [LLM-OUTPUT] SQL 생성 완료 | sql_length=54
[tc-002] [NL2SQL-1b] [LLM-OUTPUT] GENERATED_SQL: SELECT COUNT(*) FROM employee WHERE status = 'active';
[tc-002] [NL2SQL-2] [VALIDATE] SQL 검증 시작
[tc-002] [NL2SQL-2] [VALIDATE] SQL 검증 성공
[tc-002] [NL2SQL-2x] [BRANCH] 분기 결정 → EXECUTE
[tc-002] [NL2SQL-3] [EXECUTE] SQL 실행 시작
[tc-002] [NL2SQL-3] [EXECUTE] SQL 실행 완료 | row_count=1 | execution_time_ms=15
[tc-002] [NL2SQL-4] [ANSWER] 답변 생성 시작 | row_count=1
[tc-002] [NL2SQL-4a] [LLM-INPUT] LLM 호출 시작 (답변 생성)
[tc-002] [NL2SQL-4b] [LLM-OUTPUT] 답변 생성 완료 | answer_length=22
[tc-002] [NL2SQL-4b] [LLM-OUTPUT] ANSWER: 현재 재직 중인 직원은 총 27명입니다.
[tc-002] [NL2SQL-5] [COMPLETE] NL2SQL 그래프 실행 완료 | has_sql=True | answer_length=22

-----------------------------------------------------------

In [49]:
# 테스트 케이스 3: JOIN 쿼리
print("\n" + "="*100)
print("테스트 케이스 3: 부서별 직원 조회")
print("="*100 + "\n")

result3 = nl2sql_graph_test.invoke(
    question="개발팀 소속 직원들을 보여주세요",
    request_id="tc-003"
)

print("\n" + "-"*100)
print("최종 결과:")
print("-"*100)
print(f"생성된 SQL:\n{result3['generated_sql']}\n")
print(f"검증 결과: {'✅ 통과' if result3['validated'] else '❌ 실패'}")
print(f"\n최종 답변:\n{result3['answer']}")
print("\n" + "="*100 + "\n")


테스트 케이스 3: 부서별 직원 조회

[tc-003] [NL2SQL-0] [INIT] NL2SQL 그래프 실행 시작 | question=개발팀 소속 직원들을 보여주세요
[tc-003] [NL2SQL-1] [GENERATE] SQL 생성 시작 | question=개발팀 소속 직원들을 보여주세요
[tc-003] [NL2SQL-1a] [LLM-INPUT] LLM 호출 시작
[tc-003] [NL2SQL-1b] [LLM-OUTPUT] SQL 생성 완료 | sql_length=121
[tc-003] [NL2SQL-1b] [LLM-OUTPUT] GENERATED_SQL: SELECT employee.*  FROM employee JOIN department ON employee.department_id = department.id WHERE department.name = '개발팀';
[tc-003] [NL2SQL-2] [VALIDATE] SQL 검증 시작
[tc-003] [NL2SQL-2] [VALIDATE] SQL 검증 성공
[tc-003] [NL2SQL-2x] [BRANCH] 분기 결정 → EXECUTE
[tc-003] [NL2SQL-3] [EXECUTE] SQL 실행 시작
[tc-003] [NL2SQL-3] [EXECUTE] SQL 실행 완료 | row_count=3 | execution_time_ms=18
[tc-003] [NL2SQL-4] [ANSWER] 답변 생성 시작 | row_count=3
[tc-003] [NL2SQL-4a] [LLM-INPUT] LLM 호출 시작 (답변 생성)
[tc-003] [NL2SQL-4b] [LLM-OUTPUT] 답변 생성 완료 | answer_length=27
[tc-003] [NL2SQL-4b] [LLM-OUTPUT] ANSWER: 개발팀에는 총 15명의 직원이 소속되어 있습니다.
[tc-003] [NL2SQL-5] [COMPLETE] NL2SQL 그래프 실행 완료 | has_sql=True | answer_length=

## 8. 실제 데이터베이스 연결 테스트 (선택사항)

In [ ]:
# 실제 DB 연결 (환경변수에 DATABASE_URL이 있는 경우)
import psycopg

database_url = os.getenv('DATABASE_URL')

if database_url:
    try:
        # 연결 테스트
        with psycopg.connect(database_url) as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT COUNT(*) FROM employee;")
                count = cur.fetchone()[0]
                print(f"✅ 데이터베이스 연결 성공!")
                print(f"Employee 테이블 레코드 수: {count}")
                
        USE_REAL_DB = True
    except Exception as e:
        print(f"❌ 데이터베이스 연결 실패: {e}")
        USE_REAL_DB = False
else:
    print("⚠️ DATABASE_URL이 설정되지 않았습니다. Mock DB를 사용합니다.")
    USE_REAL_DB = False

In [ ]:
# 실제 DB를 사용한 SQL 실행 함수
if USE_REAL_DB:
    def execute_sql_real(sql: str) -> Dict[str, Any]:
        """실제 데이터베이스에서 SQL 실행"""
        import time
        
        start_time = time.time()
        
        with psycopg.connect(database_url) as conn:
            with conn.cursor() as cur:
                cur.execute(sql)
                rows = cur.fetchall()
                columns = [desc[0] for desc in cur.description]
        
        execution_time_ms = int((time.time() - start_time) * 1000)
        
        return {
            "columns": columns,
            "rows": rows,
            "row_count": len(rows),
            "execution_time_ms": execution_time_ms
        }
    
    print("✅ 실제 DB SQL 실행 함수 정의 완료")
    
    # 실제 DB를 사용한 테스트
    print("\n" + "="*100)
    print("실제 데이터베이스 테스트")
    print("="*100 + "\n")
    
    test_sql = "SELECT emp_id, name, position FROM employee LIMIT 5;"
    real_result = execute_sql_real(test_sql)
    
    print(f"실행된 SQL: {test_sql}")
    print(f"\n결과: {real_result['row_count']}개 행")
    print(f"실행 시간: {real_result['execution_time_ms']}ms")
    print(f"\n데이터:\n{json.dumps(real_result, indent=2, ensure_ascii=False, default=str)}")
else:
    print("실제 DB가 연결되지 않아 이 섹션을 건너뜁니다.")

## 요약

이 노트북에서는 다음을 테스트했습니다:

1. ✅ **환경 설정**: LLM, Mock 스키마, Mock 데이터 준비
2. ✅ **LLM 연결**: 실제 OpenAI API 또는 Mock LLM
3. ✅ **SQL 생성 노드**: 자연어 → SQL 변환
4. ✅ **SQL 검증 노드**: 보안 검증 (SELECT만 허용, 위험 키워드 차단)
5. ✅ **SQL 실행 노드**: Mock 또는 실제 DB 실행
6. ✅ **답변 생성 노드**: SQL 결과 → 자연어 답변
7. ✅ **전체 Graph**: LangGraph 워크플로우 통합 테스트

### 다음 단계

- 다양한 질문 패턴으로 테스트 확장
- 에러 케이스 테스트 추가
- 성능 측정 및 최적화
- 실제 프로젝트 코드와 통합